In [2]:
import pandas as pd
import numpy as np

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, GridSearchCV

In [3]:
data = load_breast_cancer()

df = pd.DataFrame(
    data.data,
    columns=data.feature_names
)

df["target"] = data.target

print(df.head())
print(df.shape)
print(df["target"].value_counts())

   mean radius  mean texture  mean perimeter  mean area  mean smoothness  \
0        17.99         10.38          122.80     1001.0          0.11840   
1        20.57         17.77          132.90     1326.0          0.08474   
2        19.69         21.25          130.00     1203.0          0.10960   
3        11.42         20.38           77.58      386.1          0.14250   
4        20.29         14.34          135.10     1297.0          0.10030   

   mean compactness  mean concavity  mean concave points  mean symmetry  \
0           0.27760          0.3001              0.14710         0.2419   
1           0.07864          0.0869              0.07017         0.1812   
2           0.15990          0.1974              0.12790         0.2069   
3           0.28390          0.2414              0.10520         0.2597   
4           0.13280          0.1980              0.10430         0.1809   

   mean fractal dimension  ...  worst texture  worst perimeter  worst area  \
0             

In [4]:
X = df.drop(columns=["target"])
y = df["target"]

print(X.shape)
print(y.shape)

(569, 30)
(569,)


In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

Training data: (455, 30)
Testing data: (114, 30)


In [6]:
knn_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier())
])

In [7]:
param_grid = {
    "knn__n_neighbors": list(range(1, 22, 2))
}

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

grid_search = GridSearchCV(
    estimator=knn_pipeline,
    param_grid=param_grid,
    cv=cv,
    scoring="accuracy",
)

grid_search.fit(X_train, y_train)

print("Best K:", grid_search.best_params_["knn__n_neighbors"])
print("Best CV accuracy:", grid_search.best_score_)

Best K: 9
Best CV accuracy: 0.9648351648351647


In [8]:
best_k = grid_search.best_params_["knn__n_neighbors"]

final_model = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(n_neighbors=best_k))
])

final_model.fit(X_train, y_train)

print("Final model trained with K =", best_k)

Final model trained with K = 9


In [9]:
y_pred = final_model.predict(X_test)

results = pd.DataFrame({
    "Actual": y_test,
    "Predicted": y_pred
})

results["Actual Label"] = results["Actual"].map({
    0: "Malignant",
    1: "Benign"
})

results["Predicted Label"] = results["Predicted"].map({
    0: "Malignant",
    1: "Benign"
})

print(results.head(10))

     Actual  Predicted Actual Label Predicted Label
256       0          0    Malignant       Malignant
428       1          1       Benign          Benign
501       0          0    Malignant       Malignant
363       1          1       Benign          Benign
564       0          0    Malignant       Malignant
464       1          1       Benign          Benign
358       1          1       Benign          Benign
343       0          0    Malignant       Malignant
516       0          0    Malignant       Malignant
567       0          0    Malignant       Malignant


# K-Nearest Neighbors (KNN) — Classification

## 1. What is KNN?

K-Nearest Neighbors (KNN) is a **supervised machine learning algorithm** used for classification and regression.

In classification, KNN predicts the class of a new data point by finding the K nearest training data points and selecting the majority class among them.

**Example:** Predict whether a tumor is malignant or benign based on measurements from nearby training samples.

## 2. How Does KNN Work?

1. Choose a value for K.
2. Calculate the distance between the new data point and every training data point.
3. Sort the distances in ascending order.
4. Select the K nearest neighbors.
5. Count the class labels of those neighbors.
6. Predict the class with the majority vote.

### Example

Suppose K = 5 and the five nearest neighbors have these labels:

- Benign: 3
- Malignant: 2

The predicted class is **Benign**, because it receives the majority of votes.

## 3. Mathematics Behind KNN

### Euclidean Distance

Euclidean distance measures the straight-line distance between two points.

For two features:

\[
d(X,A)=\sqrt{(x_1-a_1)^2+(x_2-a_2)^2}
\]

For n features:

\[
d(X,A)=\sqrt{\sum_{i=1}^{n}(x_i-a_i)^2}
\]

Where:
- X = new data point
- A = training data point
- n = number of features

The algorithm calculates distances to the training samples and identifies the nearest neighbors.

### Majority Voting

For classification, the predicted class is the most frequent class among the K nearest neighbors.

KNN can also use distance-weighted voting, where closer neighbors receive greater influence.

## 4. How Do We Choose K?

K is a hyperparameter that specifies the number of neighbors considered during prediction.

- **Small K:** Sensitive to noise and may overfit.
- **Large K:** Considers more neighbors and may underfit.
- **Best K:** Depends on the dataset and is usually selected using cross-validation.

Odd values are often convenient for binary classification because they can reduce tied votes, although ties can still occur.

The heuristic \(K\approx\sqrt{n}\), where n is the number of training samples, can be used as a starting point. It does not guarantee the best K.

## 5. Why Is Feature Scaling Important?

KNN relies on distances. If features have very different numerical ranges, features with larger ranges can dominate the distance calculation.

For example, salary values may range from 10,000 to 100,000, while age values may range from 18 to 60.

StandardScaler transforms each feature using:

\[
z=\frac{x-\mu}{\sigma}
\]

Where:
- x = original value
- μ = mean of the feature
- σ = standard deviation of the feature

We fit the scaler using training data and use the same fitted scaler to transform test data.

```python
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
```

**Important:** Never fit the scaler on the test data. This would introduce data leakage.

## 6. Train-Test Split

We first divide the dataset into training and testing sets.

For example, with 100 rows and an 80/20 split:

- 80 rows are used for training and model selection.
- 20 rows are kept aside for final testing.

The test set must remain untouched while choosing K.

```python
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)
```

`stratify=y` attempts to preserve the target-class proportions in both sets.

## 7. What Is Cross-Validation?

Cross-validation evaluates a model using different portions of the training data.

### Example: 5-Fold Cross-Validation

Suppose the training set contains 400 rows.

- Divide the training data into 5 folds.
- Each fold contains approximately 80 rows.
- In each round, use 320 rows for training and 80 rows for validation.
- Repeat for 5 rounds, allowing each fold to serve as the validation set once.
- Calculate the average validation score.

\[
\text{Mean CV Score}
=\frac{\sum_{i=1}^{k}\text{Score}_i}{k}
\]

Here, k is the number of folds.

Cross-validation helps us select K based on performance across multiple splits rather than relying on a single split.

### StratifiedKFold

```python
from sklearn.model_selection import StratifiedKFold

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)
```

- `n_splits=5`: Creates five folds.
- `shuffle=True`: Shuffles the rows before dividing them.
- `random_state=42`: Makes the randomized split reproducible.

StratifiedKFold attempts to preserve class proportions in each fold.

The `cv` variable stores the splitter configuration. It does not store trained models or validation scores.

## 8. What Is Hyperparameter Tuning?

Hyperparameters are settings chosen before fitting the final model.

For KNN, examples include:

- `n_neighbors`: Number of neighbors.
- `weights`: How neighbors contribute to voting.
- `metric`: Distance metric used to identify neighbors.

We try different hyperparameter values and select those that perform well on validation data.

## 9. What Is GridSearchCV?

GridSearchCV automates hyperparameter tuning.

For example:

```python
param_grid = {
    "knn__n_neighbors": list(range(1, 22, 2))
}
```

This generates:

`1, 3, 5, 7, 9, 11, 13, 15, 17, 19, 21`

GridSearchCV evaluates every specified K value using the chosen cross-validation strategy.

```python
grid_search = GridSearchCV(
    estimator=knn_pipeline,
    param_grid=param_grid,
    cv=cv,
    scoring="accuracy"
)
```

- `estimator`: Model or pipeline being evaluated.
- `param_grid`: Hyperparameter combinations to try.
- `cv`: Cross-validation strategy.
- `scoring`: Metric used to compare the candidates.

GridSearchCV selects the candidate with the best mean cross-validation score. By default, it then refits the best estimator on the full training data.

**Grid search automates hyperparameter selection; cross-validation evaluates each candidate across multiple folds.**

### RandomizedSearchCV

RandomizedSearchCV is an alternative that samples a specified number of hyperparameter combinations instead of exhaustively evaluating every specified combination.

- GridSearchCV: Tests all specified combinations.
- RandomizedSearchCV: Tests a selected number of sampled combinations.

## 10. What Is a Pipeline?

A Pipeline connects multiple preprocessing and modeling steps into one object.

```python
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier

knn_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(n_neighbors=5))
])
```

When we call:

```python
knn_pipeline.fit(X_train, y_train)
```

The pipeline:

1. Fits StandardScaler on the training features and transforms them.
2. Passes the scaled features and target labels to KNN.
3. Fits the KNN estimator.

When we call:

```python
knn_pipeline.predict(X_test)
```

The pipeline transforms the test features using the scaler fitted during training, then makes predictions using KNN.

A pipeline is useful for reducing repetitive code and ensuring preprocessing is applied consistently.

During cross-validation, a pipeline also ensures that the scaler is fitted separately within each training fold, helping prevent data leakage.

## 11. Complete KNN Workflow

1. Load the dataset.
2. Inspect the rows, columns, and target classes.
3. Separate features X and target y.
4. Split the data into training and test sets.
5. Scale features.
6. Use cross-validation to select a suitable K.
7. Fit the final KNN model using the selected K and the full training set.
8. Predict classes for the test set.
9. Evaluate the final model using classification metrics.

The final test set must not be used to select K or tune the model.

## 12. Important Characteristics of KNN

- KNN is a supervised learning algorithm.
- It is a non-parametric algorithm.
- It is often called a lazy learning algorithm because fitting mainly stores the training examples rather than learning model coefficients.
- Distance calculations are performed when predictions are requested.
- Feature scaling is generally important.
- Prediction can be expensive for large datasets because distances to training examples must be considered.
- The choice of K and distance metric affects predictions.

## Key Takeaway

KNN predicts a class using the labels of nearby training samples. We scale features so distances are meaningful, use cross-validation to choose a suitable K, fit the final model on all training data, and evaluate it on an untouched test set.
